# GEMS M8 — Physics-guided probabilistic fault discovery

**Central question.** Can physically meaningful potential-field signals, Monte Carlo stability and fault-network completion recover withheld fault systems that B0 / B1 and the M7 generator miss — at a manageable prediction density — and does that become a reproducible gain in geographically validated, official-style DTI?

Recorded M7 offline values (not leaderboard results): `disc_official` DTI B0 raw 0.104, B1 raw 0.092, fusion 0.123, + skeleton 0.128, + crest thinning 0.162, + 300 m ring 0.198. The M7 candidate pool covered ~4.2 % of withheld systems (5 / 120) and 0 of the 93 systems missed by both CNNs.

```text
 Magnetic / Gravity     Terrain / DEM        Known fault map
        |                   |                     |
 potential-field ops   surface morphology    network completion
        +-------------------+---------------------+
                            |
            HIGH-RECALL CANDIDATES (unions, provenance)
                            |
           MONTE CARLO STABILITY / UNCERTAINTY
                            |
        GEOLOGICAL FEATURES  (CNN support + physics + graph context)
                            |
                    CANDIDATE RANKING
                            |
             UNCERTAINTY-AWARE RASTERISATION
                            |
               CROSS-FITTED FUSION -> GEOTIFF
```

**Rules of this notebook**
* No CNN is retrained. B0 / B1 maps and every M7 artifact are reused read-only; M8 writes only under `outputs/m8_physics_probabilistic/`.
* One protocol for every number (official kernel / α / β, known faults masked pixel-exactly, eligible-pixel budget). Every label-dependent choice for a zone is made on the other zones. These zones were inspected before: **not an untouched test set**.
* Profiles (`PILOT` default, `VALIDATE`, `FULL`) change computational scope only — never units, masks, metric, budget, operators or gate thresholds.
* Oracle numbers read the truth: diagnostic upper bounds that decide whether ranking is worth running; they never promote a model.
* **No number exists until the cells run.** Every markdown interpretation below describes how to read possible outcomes; none reports a measurement.

| section | question | gate |
|---|---|---|
| 00 | inputs present and compatible? | stops on CRS / template mismatch |
| 01 | baseline reference under one protocol | compatibility table |
| 02 | where does geological information disappear? | (G1 after 06) |
| 03 / 03b | physics operators, multiscale worms | — |
| 04 | magnetic–gravity, surface–subsurface, context | — |
| 05 | fault-normal profiles | — |
| 06 | high-recall candidates + oracle | **G1, G2** (ranking stops on NO_GO) |
| 07 | Monte Carlo stability | G3 |
| 08 | stochastic network completion | — |
| 09 | DEM pilot increment | G4 |
| 10–13 | features, ranking, (graph), rasterisation | G5 |
| 14 | Systems A / B / C, fusion, verdicts | **G6** |
| 15 | validated GeoTIFFs (never uploaded) | — |

Code: `gems_bench/m8/` · configuration: `configs/m8/physics_probabilistic_discovery.yaml` · guide: `docs/M8_PHYSICS_PROBABILISTIC_DISCOVERY.md`.

## 00. Setup, artifact recovery and compatibility

**Research question.** Are the raw grid, template, folds, splits, B0 / B1 maps and M7 artifacts present and mutually compatible?

**Method.** Mount Drive, install the repository, open the pipeline context (cached data stages are validated and staged, never rebuilt), create `m8 = pipe.m8(M8_CONFIG)`. M8 owns a read-only M7 engine; its GEMS3 engine writes only under `outputs/m8_physics_probabilistic/gems3_engine`.

**Success / failure criterion.** A CRS or template mismatch stops the notebook. Missing optional bands only disable the operators that need them (logged in `manifests/band_availability.csv`).

### Central configuration
`M8_CONFIG` overrides the YAML; unknown keys raise an error naming the closest valid key. `shared` is the M7-schema protocol block (validated by the M7 loader). Expensive / GPU stages are off by default; the default runtime is CPU.

In [ ]:
CONFIG = {                                         # master-pipeline context (data stages are only validated / staged)
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'gems3-discovery-research',               # branch or commit SHA
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',
    'run_mode': 'PREPARE_DATA',                      # data stages reuse Drive; M8 never trains CNNs
    'failure_policy': 'interactive',
    'download_missing': False,
    'run_advanced_models': False, 'train_baselines': [], 'run_full_inference': False, 'run_submission': False,
    'feature_stage': 'lazy',
    'use_gpu': False,                                # PILOT is CPU-only
    'seed': 2026,
}
DEM_CONFIG = {'enabled': False, 'mode': 'inventory_only', 'download': False}
PRIVATE_REPO = False

M8_CONFIG = {
    'profile': 'PILOT',                              # PILOT | VALIDATE | FULL (computational scope only)
    'profile_overrides': {},                         # e.g. {'mc_realizations': 32}: scope keys only
    'overwrite_existing': False,                     # True recomputes M8 stages (immutable tables are kept)
    'sections': {                                    # independent switches
        's01_baseline': True,                        # baseline recovery
        's02_signal_recovery': True,                 # signal-recovery diagnostics
        's03_potential_fields': True,                # potential-field transformations
        's03b_structural_worms': True,               # multiscale worms
        's04_multimodal': True,                      # magnetic-gravity alignment, surface-subsurface, context
        's05_fault_normal_profiles': True,           # fault-normal profiles
        's06_candidates': True,                      # candidate generation + oracle gate
        's07_monte_carlo_signal': True,              # Monte Carlo signal perturbation
        's08_monte_carlo_completion': True,          # Monte Carlo network completion
        's09_dem_pilot': True,                       # DEM pilot (never blocks other sections)
        's10_candidate_features': True,              # candidate features
        's11_ranking': True,                         # LightGBM ranking
        's12_graph': False,                          # probabilistic graph experiment (optional)
        's13_rasterization': True,                   # probabilistic rasterisation
        's14_ensemble': True,                        # cross-fitted fusion + gates + report
        's15_submission': False,                     # submission generation (never uploads)
    },
    'shared': {                                      # M7-schema protocol (units, budget, PU labels, promotion rules)
        'models': ['b0_reference', 'b1_bg_weight'],
        'submission': {'source': 'oof_stitched', 'area_frac_source': 'geo'},   # 'fold_average' needs a GPU
        'dem': {'download': False},
    },
    'bands': {'verified': {}, 'units': {}},          # confirmed derivative definitions / units (else PROVISIONAL)
    'monte_carlo': {'measured_noise': {}},           # {family: sigma} -> measured uncertainty instead of heuristic
    'dem_pilot': {'reuse_m7': True, 'stream': False},
    'ranking': {'force': False},
}
SHOW_FIGURES = True                                  # figure display (saving is controlled by save_figures)

In [ ]:
import base64, os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = f"https://github.com/{CONFIG['repo']}.git"

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN') or ''
    hdr = base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {hdr}'}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args[:3])} failed:\n{r.stderr.strip()[-1500:]}")
    return r

if not os.path.exists(f'{CODE}/.git'):
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'remote', 'set-url', 'origin', REPO_URL)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
before = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if git('-C', CODE, 'rev-parse', '--verify', '-q', f'refs/remotes/origin/{ref}', check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', ref)
    git('-C', CODE, 'merge', '-q', '--ff-only', f'origin/{ref}')
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{CODE}[colab]'], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
print('code @', HEAD)
if 'gems_bench' in sys.modules and HEAD != before:
    raise RuntimeError('Code changed after gems_bench was imported: Runtime -> Restart session, then run from the top.')

In [ ]:
import json, pathlib
import numpy as np, pandas as pd
from IPython.display import Image, Markdown, display
from gems_bench.pipeline import Pipeline

pipe = Pipeline.from_notebook(CONFIG, DEM_CONFIG)
pipe.step('bootstrap')
for s in ('raw_data', 'native_grid', 'spatial_folds', 'discovery_splits', 'feature_store'):
    pipe.step(s)                                     # validated on Drive and staged locally (never regenerated)

m8 = pipe.m8(M8_CONFIG)
OUT = m8.ws.root
print('profile:', m8.profile)
print('M8 outputs (local):', OUT)
print('M8 outputs (Drive):', m8.cfg['_drive_root'])
print('M7 artifacts (read-only):', m8.m7.ws.root)

def show(names, sub):
    if not (SHOW_FIGURES and m8.cfg.get('display_figures', True)):
        return
    for n in names:
        p = OUT / sub / f'{n}.png'
        display(Image(filename=str(p))) if p.exists() else print('not produced:', sub, n)

def tab(name, sub, n=30):
    t = m8.ws.load_table(name, sub)
    display(t.head(n)) if len(t) else print('empty / not produced:', sub, name)
    return t

In [ ]:
setup = m8.s00_setup()                    # read-only; raises if CRS / template checks fail
print(json.dumps({k: v for k, v in setup['checks'].items() if k != 'bands'}, indent=1, default=str))
display(setup['bands'])
display(setup['inventory'])               # integration inventory: reused functions, files, consumers, assumptions
display(m8.m7.protocol_keys())

## 01. Recover the baselines and establish the reference

**Research question.** Under the shared protocol, what are B0, B1, their rank fusion, crest / skeleton thinning and the 300 m ring worth — and are the M7 artifacts compatible with today's grid, folds, splits, models and metric?

**Geological rationale.** A new geological signal matters only if it beats the strongest existing model on the same evaluation; historical numbers came from several protocols.

**Hypothesis.** The M7 nested selection is reusable (protocol keys match) and System A = fusion + crest thinning remains the strongest ring-free method.

**Method.** Reuse M7's `baseline_selection.json` only when the protocol keys stored in its immutable table equal the current units' keys; otherwise recompute with M7's code. Rebuild every per-zone held-out map (`validation/maps/s01_*`), reload / recompute the per-system B0 / B1 recovery classes, run the compatibility checks (CRS, extent, transform, nodata, fold and split ids, model configuration, map fingerprints, metric constants, budget domain, score-map processing).

**Planned diagnostic.** `baseline/compatibility.csv`, `recorded_vs_this_run.csv` (the `source` column states whether values were recomputed here), `baseline_dti.png`, `baseline_manifest.json`.

**Success / failure criterion.** No compatibility row `DIFFERS`. A difference to recorded M7 values is reported, never reconciled, and never claimed as a reproduction unless `source` says recomputed.

**How to read the outcome.** If keys differ, every later comparison uses the recomputed selection and M7 numbers are context only. If the ring variant is best, remember the proxy under-represents new faults next to known traces.

In [ ]:
if m8.enabled('s01_baseline'):
    s01 = m8.s01_baseline()
    t = s01['table']
    display(t[[c for c in ('task', 'method', 'dti_crossfit', 'fold_min', 'optimism', 'area_frac_eligible_mean',
                           'dti_zone0', 'dti_zone1', 'dti_zone2') if c in t]])
    display(s01['recorded'])
    comp = s01['compatibility']
    display(comp)
    if len(comp) and (comp.status == 'DIFFERS').any():
        print('WARNING: compatibility differences:', comp[comp.status == 'DIFFERS'].check.tolist())
    print(json.dumps(m8.read_json('baseline/baseline_manifest.json'), indent=1, default=str)[:3000])
    show(['baseline_dti'], 'baseline')

## 02. Where does geological information disappear? — the first decisive experiment

**Research question.** For each withheld fault system, at which step of the extraction chain does its signal stop: raw field → structural gradient → normalised response → directional edge → crest → gated fragment → persistence filter → linked polyline → ranked candidate?

**Geological rationale.** A fault is visible in potential fields only when it juxtaposes contrasting rocks or offsets a basement surface; it can then be lost by smoothing, global thresholds, coherence gates, short-length filters or ranking. Each loss has a different remedy.

**Hypothesis.** A substantial share of systems missed by both CNNs still carries a raw response that the M7 chain discards (an extraction bottleneck), rather than no response at all.

**Method.** Per system and modality (magnetic, gravity, terrain, conductivity, basement): three raw-response definitions (cross-strike contrast, gradient percentile, local / regional variance) each tested against a **system-matched null** (the same trace rigidly translated 3–12 km to valid ground away from mapped faults); stages 2–9 as coverage within one pixel. Failure reason = first stage lost. All label use is diagnostic.

**Planned diagnostic.** `signal_recovery/stage_matrix`, `reasons` (failure_reason / failure_category), `stage_recall` (system- and length-weighted, by length, strike, region, B0/B1 class), `missed_by_both_by_stage`, `stage_density`, `stage_funnel.png`, 8-panel case studies for the eight required classes.

**Success / failure criterion.** Descriptive: the section succeeds when each system has a reason. It feeds Gate 1 (evaluated after section 06).

**How to read the outcome.** Many `raw_present -> crest_missing` / `fragment_missing` rows → fix extraction (local thresholds, operators). Many `persistent -> linked_missing` → length / linking filters. `linked -> ranked_missing` → ranking / density. `raw_response_absent` under all definitions → information limit at 100 m *for these operators* (DEM or other data may help); it is not proof of invisibility.

In [ ]:
if m8.enabled('s02_signal_recovery'):
    s02 = m8.s02_signal_recovery()
    sub = 'signal_recovery'
    tab('summary', sub)
    r = m8.ws.load_table('reasons', sub)
    if len(r):
        display(r.groupby(['task', 'failure_reason']).size().rename('n_systems').reset_index()
                .sort_values('n_systems', ascending=False).head(20))
    st = m8.ws.load_table('stage_recall', sub)
    if len(st):
        display(st[(st.stratum == 'all')].pivot_table(index=['task', 'modality'], columns='stage', values='system_recall'))
    tab('missed_by_both_by_stage', sub, 40)
    show(['stage_funnel'], sub)
    idx = tab('case_study_index', f'{sub}/case_studies', 16)
    for f in idx.figure.head(8) if len(idx) else []:
        show([f[:-4]], f'{sub}/case_studies')

## 03. Physics-guided potential-field features

**Research question.** Which balanced potential-field operators expose fault-related edges that plain gradients bury under strong regional sources?

**Geological rationale.** Total horizontal gradient peaks over steep contacts but is dominated by strong, shallow sources. Tilt, TDX and theta normalise by the vertical derivative / analytic signal so weak and strong edges have comparable amplitude; local contrast normalisation does the same spatially. Analytic signal is depth- but not magnetisation-direction-sensitive. Gravity resolution (~2 km) limits its useful scales.

**Hypothesis.** Balanced operators (tilt edge, TDX, LCN) put crests on low-contrast faults that THDR misses, especially in quiet basins.

**Method.** Competing definitions at 100–3200 m (skipped below half the configured effective resolution): THDR, analytic signal, tilt edge, TDX, theta, LCN, and *_band variants from `tmi_hg / tmi_vg`, `iso_grav_anom_hg / _vg` only when the derivative audit marks them verified / consistent. Singularities: tilt / TDX / theta undefined where the analytic signal is below a low quantile; denominators floored; nodata never propagates.

**Planned diagnostic.** `band_provenance.csv` (VERIFIED_METADATA / CONSISTENT_EMPIRICAL / PROVISIONAL), `effective_resolution.csv` (configured vs spectral), `operator_inventory.csv` (allowed / refused + reason + units), selected GeoTIFFs.

**Success / failure criterion.** Layers are label-free inputs; their value is judged in sections 06 (coverage) and 11 (ranking ablations).

**How to read the outcome.** PROVISIONAL derivative bands are simply not used inside physical formulas. A spectral resolution much coarser than configured means the smallest scales add noise, not information.

In [ ]:
if m8.enabled('s03_potential_fields'):
    s03 = m8.s03_potential_fields()
    sub = 'geophysical_features'
    tab('band_provenance', sub)
    tab('effective_resolution', sub)
    op = m8.ws.load_table('operator_inventory', sub)
    if len(op):
        display(op.groupby(['family', 'operator', 'allowed', 'reason']).size().rename('n_scales').reset_index())
    tab('saved_layers', sub)

## 03b. Multiscale geological worms

**Research question.** Do edges that persist through scale-space (upward continuation) recover faults that a fixed-scale lineament extractor misses?

**Geological rationale.** Upward continuation is the physical scale-space of a potential field: deeper sources dominate at greater heights. A worm that persists marks a source edge with depth extent (fault-like); its lateral migration with height indicates dip. Terrain uses Gaussian scale-space.

**Hypothesis.** Persistent worms are rarer than single-scale crests but cover a larger share of withheld systems per kilometre.

**Method.** Per level: THDR, NMS, local-percentile threshold (20 km blocks), coherence gate; base worm pixels linked to each coarser level within a tolerance and strike difference. Statistics: persistence, consecutive run, orientation stability, ridge stability, scale of maximum support, migration, continuity.

**Planned diagnostic.** `worm_summary.csv`, `worms_<family>.tif`, `coverage_vs_fixed_scale.csv` (evaluation: persistent worms vs all base-level crests, with densities).

**Success / failure criterion.** Worms are worth keeping when persistent worms reach comparable system recall at clearly lower density than base crests; final value is measured in section 06 / 11.

**How to read the outcome.** Similar recall at lower density = persistence is a useful *feature*; much lower recall = faults here are shallow / single-scale and persistence must not be a filter (M8 keeps it a feature).

In [ ]:
if m8.enabled('s03b_structural_worms'):
    s03b = m8.s03b_structural_worms()
    sub = 'geophysical_features/structural_worms'
    tab('worm_summary', sub)
    tab('coverage_vs_fixed_scale', sub)
    show(['worms_vs_fixed_scale'], sub)

## 04. Multimodal structural reasoning

**Research question.** Where do magnetic and gravity edges agree within a physically justified tolerance, and which structures are surface-expressed versus potentially buried?

**Geological rationale.** A fault juxtaposing different rocks produces both edges, but their maxima need not coincide (different source depths, dip, gravity resolution). Terrain lineaments may be faults, erosional or depositional features: never confirmations by themselves. Strain-rate and seismicity are regional descriptors; a scalar strain magnitude gives no fault orientation, and low seismicity never rejects a fault.

**Hypothesis.** Tolerant agreement (≤ 800 m) captures many more shared boundaries than exact coincidence, and agreeing edges are enriched in mapped faults.

**Method.** Nearest-worm matching with distance and strike tolerances in both directions; crest classes (magnetic only / gravity only / agreeing / discordant); surface–subsurface classes; regional percentiles of context bands.

**Planned diagnostic.** `mag_grav_agreement.csv` (agreement share, offsets, discordance by tolerance), `surface_subsurface.csv`, `classes_by_region.csv`, rasters.

**Success / failure criterion.** Label-free layers; their value is judged by the coverage of the `xmod_agree` family (06) and the agreement features (11).

**How to read the outcome.** A large discordant share near agreeing crests suggests contacts of different geometry, not one fault; strongly region-dependent agreement argues for region-aware budgets.

In [ ]:
if m8.enabled('s04_multimodal'):
    s04 = m8.s04_multimodal()
    sub = 'geophysical_features/multimodal'
    tab('mag_grav_agreement', sub)
    tab('surface_subsurface', sub)
    tab('classes_by_region', sub, 40)
    tab('context_layers', 'geophysical_features/context')

## 05. Fault-normal geophysical profiles

**Research question.** Do candidate traces carry a consistent, localised step across their strike — like mapped faults — or only a broad gradient?

**Geological rationale.** A fault juxtaposing contrasting units gives a step whose sign stays constant along strike and which is localised on the trace; a lithological contact or regional gradient gives broad, sign-variable or diffuse transitions.

**Hypothesis.** Mapped faults show higher sign consistency, continuity and localisation than random controls; unlabelled geophysical boundaries are intermediate (some are unmapped faults).

**Method.** Smoothed curved paths, stations every 2 px, consistently oriented normals, bilinear samples at ±100…1200 m on magnetic, gravity, conductivity, terrain and basement fields; samples outside the unit's evaluation zone or near nodata are dropped (no transect crosses a fold / split boundary).

**Planned diagnostic.** Per-candidate summaries (`fault_normal_profiles/units/*`), raw transects for a capped number of candidates, gallery `gallery_profiles.csv` + `gallery_contrast_vs_offset.png` for mapped faults / unlabelled boundaries / random controls / candidates.

**Success / failure criterion.** Profile features enter section 11; the gallery is descriptive.

**How to read the outcome.** Unlabelled boundaries are **not** negatives. If mapped faults and controls look alike, profile contrasts will not help ranking at 100 m.

*Order note:* before section 06 has run, profiles describe the M7 pool; section 06 re-runs this section on the M8 pool automatically.

In [ ]:
if m8.enabled('s05_fault_normal_profiles'):
    s05 = m8.s05_fault_normal_profiles()
    if s05 and 'pool_source' in s05:
        print('pool profiled per unit:', s05['pool_source'])
    tab('gallery_profiles', 'fault_normal_profiles/gallery', 40)
    show(['gallery_contrast_vs_offset'], 'fault_normal_profiles/gallery')

## 06. High-recall candidate generation — mandatory oracle gate (Gates 1 and 2)

**Research question.** Does a union of physics, terrain, cross-modal, CNN, CNN-disagreement, network-continuation and DEM families cover substantially more withheld systems than the M7 pool at the same candidate density?

**Geological rationale.** Faults express themselves through different modalities in different settings; requiring every modality would discard most of them. Recall comes from unions; specificity from ranking on modality agreement.

**Hypothesis.** Local thresholds, preserved short fragments, balanced operators and worms raise system recall at matched density, including systems missed by both CNNs.

**Method.** Label-free families on the full grid (cached), per-unit honest CNN / disagreement crests, known-tip extensions, section-08 completion hypotheses and DEM crests; multiple linking distances / curvature limits; union → deduplication with provenance (`sources`, `families`, `member_uids`). Oracle (reads truth): coverage at label-free density caps, regional density budget, kd subsets, recall by family, missed-by-both coverage, unique coverage of the lowest-confidence tercile, and the M7 pool at the same densities.

**Planned diagnostic.** `oracle/gate2.csv`, `pooled_curve.csv` (M8 vs M7), `pooled_groups.csv`, `incremental.csv`, `low_conf.csv`, `recall_density_m8_vs_m7.png`, `signal_recovery/gate1.csv`.

**Success / failure criterion.** Gate 2 GO: system recall at the gate density ≥ 0.30 AND (gain over M7 at the same density ≥ 0.05 OR missed-by-both coverage ≥ 0.10). Gate 1 PASS: ≥ 3 systems (≥ 5 %) covered by M8 physics and not by M7 linked candidates. On NO_GO, sections 10–13 refuse to run.

**How to read the outcome.** Oracle values are upper bounds, never scores. If Gate 1 fails with mostly raw-absent systems, the limit is the data at 100 m; with mostly raw-present systems, extraction still loses them.

In [ ]:
if m8.enabled('s06_candidates'):
    s06 = m8.s06_candidates()                # builds the section-08 hypotheses first (own stage)
    sub = 'candidate_generation/oracle'
    g2 = tab('gate2', sub)
    tab('gate1', 'signal_recovery')
    pc = m8.ws.load_table('pooled_curve', sub)
    if len(pc):
        display(pc.pivot_table(index=['task', 'density_cap'], columns='pool', values='system_recall'))
    pg = m8.ws.load_table('pooled_groups', sub)
    if len(pg):
        display(pg[pg.group == 'family'].sort_values(['task', 'system_recall'], ascending=[True, False]).head(30))
    tab('incremental', sub, 30)
    tab('low_conf', sub)
    show(['recall_density_m8_vs_m7'], sub)
    print('GATE 2 (primary task):', m8.gate('gate2', sub))
    if m8.enabled('s05_fault_normal_profiles'):
        m8.s05_fault_normal_profiles()       # profiles of the M8 pool (fingerprint follows section 06)

## 07. Monte Carlo geophysical uncertainty (Gate 3)

**Research question.** Which candidates are stable under physically plausible perturbations of the fields and of the extraction choices, and does the perturbation ensemble reveal structures the deterministic pool misses?

**Geological rationale.** Edge positions depend on survey noise, registration, filter scale and thresholds; a fault-related edge should survive moderate perturbation, while noise-driven or threshold-driven crests flicker.

**Hypothesis.** Stable candidates are enriched in mapped faults; occupancy crests outside the pool add a few systems at matched density.

**Method.** Per realisation (seeded): correlated noise and one registration shift per survey family (base fields only — derived bands of one survey are never perturbed independently), scale jitter, operator choice, threshold / coherence / linking / persistence choices; the section-06 pipeline extracts lines; tolerant matching to the pool (fragments of one structure count once). Distributions are recorded in `assumptions.json`; without `measured_noise` this is a **heuristic sensitivity analysis**. Checkpointed and resumable.

**Planned diagnostic.** `stability.csv` (detection frequency, positional / orientation / length variability, fragmentation, cross-modal stability, per-family frequency), `stability_vs_truth.csv` (stable / intermediate / unstable bins: share near truth, share far, systems), `mc_only_coverage.csv`, `occupancy.tif`, `occupancy_atlas.png`.

**Success / failure criterion.** Gate 3 USE_IN_MODEL if MC-only structures add ≥ 2 systems at matched density OR (after section 11) `physics_mc` beats `physics_only` by ≥ 0.003 DTI; otherwise MC is kept for uncertainty reporting only.

**How to read the outcome.** Detection frequency is processing stability, **not** a calibrated fault probability. If stable and unstable bins have similar near-truth shares, stability will not help ranking; do not hard-filter unstable candidates without that evidence.

In [ ]:
if m8.enabled('s07_monte_carlo_signal'):
    s07 = m8.s07_monte_carlo_signal()       # PILOT: 16 realisations on one discovery unit
    sub = 'monte_carlo/stability'
    st = m8.ws.load_table('stability', sub)
    if len(st):
        display(st.describe().T)
    tab('stability_vs_truth', sub)
    tab('mc_only_coverage', sub)
    tab('gate3', sub)
    print(json.dumps(m8.read_json('monte_carlo/realizations/assumptions.json'), indent=1, default=str)[:2500])
    show(['occupancy_atlas'], 'monte_carlo/uncertainty_maps')

## 08. Monte Carlo fault-network completion

**Research question.** Can evidence-conditioned stochastic continuation, gap bridging and trace relocation recover missing mapped geometry better than the deterministic M7 completers?

**Geological rationale.** Fault traces end where mapping stopped, not where faults stop; continuations follow local strike, curvature limits and geophysical support, and branch into neighbouring traces.

**Hypothesis.** Stochastic continuations recover more truncated endpoints and gaps than straight / rotated extensions at equal false length.

**Method.** Particles from each endpoint: softmax heading choice from evidence (physics layers + optional DEM), curvature penalty, orientation-field alignment, cone limit; evidence-terminated length; joins to known traces = branching support; gap bridges with attraction; displaced-trace hypotheses. Occupancy, consensus path, diversity, displacement, continuity. Evaluated (1) on the M7 corruption benchmark (systems split by fold **before** damage; completer sees the damaged map only), per class (endpoint truncation, interior gap, lateral shift, branch, incomplete network, short fault), parameters nested; (2) on disc units as System C (eligible budget, nested frac).

**Planned diagnostic.** `benchmark/benchmark_selection.csv` (DTI, recall, distance to truth, direction agreement, added false length, delta vs best M7 completer), `system_c_summary.csv`, `coverage_audit.csv`.

**Success / failure criterion.** Stochastic completion is useful where `delta_vs_best_m7 > 0` with no increase in added false length; System C enters the section-14 fusion. Completion hypotheses are never promoted to full-region predictions automatically.

**How to read the outcome.** Pseudo-discovery withholds whole systems, so near-known extensions are under-represented here: benchmark (1) is the more relevant evidence for completion.

In [ ]:
if m8.enabled('s08_monte_carlo_completion'):
    s08 = m8.s08_monte_carlo_completion()
    tab('coverage_audit', 'fault_networks', 30)
    tab('system_c_summary', 'fault_networks')
    tab('benchmark_selection', 'fault_networks/benchmark', 40)
    show(['completion_benchmark'], 'fault_networks/benchmark')

## 09. DEM pilot and incremental value (Gate 4)

**Research question.** Does 1 m surface morphology recover withheld systems that the 100 m geophysics and the CNNs do not, at matched candidate density?

**Geological rationale.** Young normal-fault scarps are surface-expressed; buried, eroded or blind faults are not. DEM can only add the former.

**Hypothesis.** Inside the pilot footprint, DEM-including candidate sets cover a few systems that their DEM-free counterparts miss.

**Method.** Reuse M7's eight-tile pilot (~2.61 GiB estimated) and mosaic read-only; optional streaming of the pilot here (`dem_pilot.stream`, needs `shared.dem.download`). Five sets on identical DEM-covered footprints at matched density: geophysics, DEM only, geophysics ∪ DEM, CNN + geophysics, CNN + geophysics + DEM. Download volume, disk use, processing time, coverage and projected full cost are logged.

**Planned diagnostic.** `dem/candidates/results.csv`, `systems.csv`, `gate4.csv`, `dem_volume_log.csv`.

**Success / failure criterion.** Gate 4 EXPAND_TARGETED if ≥ 3 systems are in the footprint, DEM adds ≥ 2 systems and the incremental system recall is ≥ 0.05; STOP otherwise; INCONCLUSIVE if the pilot holds too few systems. A DEM failure never blocks other sections.

**How to read the outcome.** DEM-covered numbers form their own protocol and are never pooled with full-zone results.

In [ ]:
if m8.enabled('s09_dem_pilot'):
    s09 = m8.s09_dem_pilot()                 # GATE_CLOSED (logged) when no mosaic exists: other sections continue
    sub = 'dem/candidates'
    tab('gate4', sub)
    tab('results', sub, 40)
    tab('dem_volume_log', sub)
    tab('pilot_tiles', 'dem/pilot')

## 10. Candidate-level feature table *(needs Gate 2 GO)*

**Research question.** Can each candidate be described by physically interpretable, provenance-tracked evidence without hiding missing information?

**Method.** One row per candidate (section-06 pool + MC-only structures): M7 descriptors (geometry, geophysics, terrain / DEM, neural, network, PU label) + M8 physics operators along the path, worm statistics, magnetic–gravity agreement and crest classes, fault-normal profiles, Monte Carlo stability, completion occupancy, context percentiles, family provenance, explicit missingness indicators (`has_mc`, `has_dem`, `has_profile`, `has_worm_*`). Labels: 1 near scored truth, 0 far, −1 unlabelled (never a negative).

**Planned diagnostic.** `feature_tables/feature_table.parquet`, `feature_groups.csv`, `feature_provenance.csv`, `missingness.csv`, `label_balance.csv`.

**Success / failure criterion.** Every feature belongs to exactly one group used by the section-11 ablations; areas without DEM / MC are explicit, never imputed silently.

In [ ]:
if m8.enabled('s10_candidate_features'):
    s10 = m8.s10_candidate_features()        # returns None (logged) when Gate 2 is closed
    tab('label_balance', 'feature_tables')
    tab('missingness', 'feature_tables', 40)
    fg = m8.ws.load_table('feature_groups', 'feature_tables')
    if len(fg):
        display(fg.groupby('group').size())

## 11. Candidate ranking (Gate 5)

**Research question.** Can a lightweight, spatially validated ranker separate faults from unrelated structural boundaries better than label-free structure and CNN support alone?

**Geological rationale.** Lithological contacts, intrusion margins and basin edges produce edges too; combinations of agreement, profile shape, persistence and network context may discriminate fault-like edges.

**Hypothesis.** `physics_cnn` or `all` beats both controls; Monte Carlo and DEM add little on their own.

**Method.** Controls `label_free`, `cnn_only`; LightGBM `physics_only`, `physics_mc`, `physics_dem`, `physics_cnn`, `all` (optional CatBoost, PU-weight sensitivity in VALIDATE / FULL). Leave one zone out; rows near the held-out zone and every row of linked networks touching it are dropped; negatives down-weighted; scored as 1-px lines at the eligible budget with nested frac. No hyper-parameter search.

**Planned diagnostic.** `ranking/ranking_summary.csv`, `training_audit.csv`, `feature_importance.csv` (descriptive), `gate5.csv`, `ranking_dti.png`.

**Success / failure criterion.** Gate 5 PASS if the best learned ranker beats max(label_free, cnn_only) by ≥ 0.005 cross-fitted DTI; otherwise model complexity is not increased (section 13 uses the better control).

**How to read the outcome.** Ranker outputs are scores, not calibrated probabilities. Gains concentrated in one zone are not robust (see section 14).

In [ ]:
if m8.enabled('s11_ranking'):
    s11 = m8.s11_ranking()
    tab('ranking_summary', 'ranking', 40)
    tab('gate5', 'ranking')
    tab('training_audit', 'ranking', 20)
    imp = m8.ws.load_table('feature_importance', 'ranking')
    if len(imp):
        display(imp.groupby('method').head(8))
    tab('gate3', 'monte_carlo/stability')     # Gate 3 updated with the ranking gain of physics_mc
    show(['ranking_dti'], 'ranking')

## 12. Optional probabilistic fragment graph *(only after Gate 5 PASS)*

**Research question.** Do geometrically plausible connections between fragments add discovery value beyond candidate-level ranking?

**Method.** Endpoint graph (distance + tangent compatibility), edge probability (heuristic, or leave-one-zone-out LightGBM on edge features), graph-propagated scores and an optional bridges variant; connectivity, false connections between true fragments, recovered structure, cross-fitted DTI.

**Success / failure criterion.** Retained only if it beats the section-11 score by ≥ 0.005 DTI; otherwise the simpler ranker stays. No large GNN.

In [ ]:
if m8.enabled('s12_graph'):
    s12 = m8.s12_graph()
    tab('graph_summary', 'graphs')
    tab('graph_metrics', 'graphs')
    tab('graph_decision', 'graphs')

## 13. Probabilistic rasterisation

**Research question.** How should ranked candidates and their Monte Carlo geometry become a scoremap on the exact 100 m grid?

**Method.** Thin centerlines; occupancy-weighted selection (budget moves to perturbation-stable positions); uncertainty-aware spreading (width from MC positional dispersion, ≤ 2 px, decaying); confidence-dependent width; crest-based selection; regional rank blend (20 km blocks) to control density; optional 300 m ring. Eligible budget; mass and non-zero area after all masks recorded; (rep, regional β, frac, ring) nested.

**Planned diagnostic.** `rasterization_summary.csv`, `rasterization_ablation.csv`, `area_mass_ledger.csv`, `representation_ablation.png`.

**Success / failure criterion.** The best representation becomes System B (`validation/maps/m8_physics`). The 300 m scoring tolerance is **not** an instruction to draw 300 m corridors: wide representations must earn their place in the ablation.

In [ ]:
if m8.enabled('s13_rasterization'):
    s13 = m8.s13_rasterization()
    sub = 'rasterization'
    tab('rasterization_summary', sub)
    tab('rasterization_ablation', sub, 40)
    tab('area_mass_ledger', sub, 20)
    show(['representation_ablation'], sub)

## 14. Ensemble and champion selection (Gate 6)

**Research question.** Do the physics system (B), network completion (C) or their cross-fitted fusion with the CNN champion (A) improve deployable official-style DTI while staying geographically robust?

**Method.** Per-zone percentile scores of A, B, C; fusions A, B, C, A+B, A+C, A+B+C on a weight simplex; weights, representation and frac chosen on the other zones. Every saved held-out map is re-scored by M7's comparison: geographic DTI, `disc_official`, retained-FP alternative, worst zone, system / length / short-fault recall, density, regional DTI, known-fault proximity, paired zone deltas, paired spatial block bootstrap; plus candidate false-length density (unlabelled, an upper bound). Verdicts: PROMOTE / RETAIN / REJECT / INCONCLUSIVE.

**Success / failure criterion.** Gate 6 PROMOTE only when a method passes every promotion rule (pooled gain ≥ 0.01, wins ≥ 2/3 of zones, worst zone ≥ −0.01, geo drop ≤ 0.01, bootstrap P(Δ > 0) ≥ 0.8, short-fault recall drop ≤ 0.02); otherwise the B0/B1 champion is retained. Oracle values never enter.

**How to read the outcome.** With three folds and three splits, paired deltas are descriptive evidence; millions of correlated pixels do not make a difference significant.

In [ ]:
if m8.enabled('s14_ensemble'):
    s14 = m8.s14_ensemble()
    tab('fusion_summary', 'ensemble', 40)
    sub = 'validation/metrics'
    tab('method_pooled', sub, 60)
    pz = m8.ws.load_table('paired_zone_deltas', sub)
    if len(pz):
        display(pz.pivot_table(index=['task', 'method'], columns='zone', values='delta_vs_ref'))
    tab('bootstrap', sub, 40)
    tab('false_length', sub, 40)
    tab('decision_verdicts', 'reports', 40)
    gates = tab('gates', 'reports')
    ans = m8.ws.load_table('research_answers', 'reports')
    for r in ans.itertuples() if len(ans) else []:
        display(Markdown(f'**{r.question}** {r.answer}  \n_{r.evidence}_'))
    show(['method_comparison', 'system_recall_comparison', 'paired_zone_deltas', 'known_proximity_recall',
          'regional_dti'], 'validation/figures')
    print('report:', OUT / 'reports' / 'M8_PHYSICS_PROBABILISTIC_REPORT.md')

## 15. Leaderboard submission files *(off by default; never uploaded)*

**Method.** Always: the CNN champion without the ring and with the optional 300 m ring at the same number of selected pixels (isolates the ring hypothesis). Only if Gate 6 PROMOTES an M8 method: the physics champion (full-region pool → features → refitted ranker → section-13 rasterisation, or the promoted fusion). Shared, corrected GEMS3 writer: template = authority (shape, CRS, transform, bounds, nodata), NaN where the template is NaN, float32 in [0, 1], eligible budget after masks, area ledger, validation that raises on failure, manifest with provenance. Existing files are never overwritten. `leaderboard_log.csv` keeps public scores (filled manually), offline proxies and oracle diagnostics in separate columns.

**Success / failure criterion.** Every written file passes validation. Respect the competition's submission limits; never tune dozens of variants on public feedback.

In [ ]:
if m8.enabled('s15_submission'):
    s15 = m8.s15_submission()
    tab('submission_index', 'submissions')
    for f in sorted((OUT / 'submissions').glob('*.json')):
        j = json.loads(f.read_text())
        if 'area_ledger' in j:
            print(f.name)
            display(pd.DataFrame(j['area_ledger']))
    display(pd.read_csv(OUT / 'submissions' / 'leaderboard_log.csv'))

## Run log, outputs and resume

Re-running any cell reuses finished stages (fingerprints in `manifests/units/`); the Monte Carlo engine resumes from its last checkpoint. After a disconnect: run the setup cells, then any section. To scale up, set `M8_CONFIG['profile'] = 'VALIDATE'` (scope only). Baselines, physics layers, worms and multimodal layers are reused, and so is the label-free candidate cache. Section 02 re-runs because VALIDATE adds the `geo` task. Section 06 re-runs because the full-region pool is in VALIDATE's scope, so everything downstream of it (Monte Carlo, completion samples, galleries, features, ranking) is recomputed. Read first: `baseline/compatibility.csv`, `signal_recovery/reasons.csv`, `candidate_generation/oracle/gate2.csv`, `reports/gates.csv`, `reports/M8_PHYSICS_PROBABILISTIC_REPORT.md`.

In [ ]:
display(pd.DataFrame(m8.log)) if m8.log else print('no stage ran in this session')
display(m8.summary())
m8.paths()
pipe.push_all()